In [25]:
# import cassio
from langchain_community.vectorstores import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
import os
from PyPDF2 import PdfReader
from dotenv import load_dotenv
from typing_extensions import Concatenate
from langchain_text_splitters import CharacterTextSplitter

In [26]:
import cassio

In [27]:
load_dotenv()

True

In [28]:
ASTRA_DB_APPLICATION_TOKEN = os.getenv("astra_token")
ASTRA_DB_ID = os.getenv("astra_id")
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")


In [29]:
pdfReader = PdfReader('../../Datasets/attention_all_you_need.pdf')

In [30]:
raw_text = ''

In [31]:
for i,page in enumerate(pdfReader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [32]:
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

In [33]:
llm = OpenAI(api_key=OPENAI_API_KEY)

In [34]:
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

In [35]:
astra_vector_store = Cassandra(
    embedding=embeddings,
    table_name='qa_demo_one',
    session=None,
    keyspace=None
    )

In [36]:
text_splitter = CharacterTextSplitter(separator='\n', chunk_size=1000, chunk_overlap=200, length_function = len)

In [37]:
texts = text_splitter.split_text(raw_text)

In [38]:
astra_vector_store.add_texts(texts)

['f1253a4ac35d442e841cc4857897024a',
 'd33eef5e5747469aa74d4ff90cb79287',
 '1d09f4adb50c4f51bd71143bbee623d8',
 '79218596b6e04b9087c95d00b9606e5f',
 '3ed5ba6e649748f2951a94a097a5a888',
 '81d69583b5044ba3aa4fd3564bff4c2b',
 '594d8ed11fd74db5816b270663a960a4',
 '85923d036db24af8be9bdfbb9573ac18',
 'c462a3b441bb4e45a2628b3f4288a6c6',
 'dd938095f5c542cd918592331faba494',
 '3cd09d13261c46edba78c46058003a1e',
 '28c817a96cfe4d2aa2ea584f97fb0de4',
 'f96201092a664402a5a7ec5c9d178eb8',
 '301a05a172e246bab859f38b40ad485a',
 '884b3c9c39a245e0b08605be4271801e',
 'ef8ee59942184b51937cf867d494a621',
 '723ccd1d4841450f8cf43e8bcc3ee783',
 '10f228f73ae141b7b7d0380cf0c2333c',
 'b0183d2e24dc43f892269d9c530a5496',
 '53c82ba25daa4dd78e584b06eb25d374',
 '7085a36de09642739f6ada027bf912d1',
 '19476dac77254aac87e36211800a7481',
 '125a6526f90c45108afb90504c2539c1',
 '66fd1d462de64d2293b9f86dbbf67029',
 'b0576cc6dfe745aeb4cf4da55337fd11',
 'e34846a228354f3d999f8b0033910bd3',
 '0a458e63d3f849c187a2a08ed19957ee',
 

In [39]:
astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

In [ ]:
first_question = True
while True:
    if first_question:
        query_text = input("Enter your Question (or type quit to exit)").strip()
    else:
        query_text = input('What your next question or type quit to exit').strip()

    if query_text.lower() =='quit':
        break
    if query_text =='':
        continue

    first_question = False
    answer = astra_vector_index.query(query_text, llm=llm)

    print(answer)
    for doc,score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print(score,doc.page_content)

 Embeddings are learned vectors that are used to represent input tokens and output tokens in a sequence transduction model. They have a dimension of dmodel and are multiplied by weight matrices in the embedding layers to transform the tokens into vectors. In the Transformer model, the same weight matrix is shared between the embedding layers and the pre-softmax linear transformation. 
0.8913132106211137 Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [ 24]. In the embedding layers, we multiply those weights bypdmodel.
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order fo